# 8.2 · External memory: sorting data bigger than RAM

*Systems and scale* · 7 exercises

Sort data that does not fit in memory by making runs and merging them, counting I/Os as you go.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-8.2) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-8.2) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-8.2)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import heapq
import math
import os
import random
import tempfile
import pathlib
import tempfile

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 8.2.1 · Make sorted runs that fit in memory  `Core`

**Task.** Write `make_runs(data, M)` that cuts the input into pieces of at most M items and sorts each piece in memory.

**Example.** `make_runs([5, 1, 9, 3, 7, 2, 8], 3)` -> `[[1, 5, 9], [2, 3, 7], [8]]`

**Big O.** O(N log M) CPU, one read and one write pass over the data.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-2-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-2-1)

In [ ]:
# 8.2.1 Make sorted runs that fit in memory
def make_runs(data, M):
    raise NotImplementedError

In [ ]:
# Test helpers: run once.
def shuffled(n, seed):
    data = list(range(n))
    random.Random(seed).shuffle(data)
    return data

In [ ]:
def test_ex_8_2_1():
    assert make_runs([5, 1, 9, 3, 7, 2, 8], 3) == [[1, 5, 9], [2, 3, 7], [8]] and make_runs([], 3) == []


test_ex_8_2_1()
print('8.2.1 passed')

### 8.2.2 · Two-way external mergesort  `Core`

**Task.** Write `two_way_sort(data, M)` that makes runs, then merges runs in pairs, pass after pass, until one run is left. Return `(sorted_list, passes)` counting the run-making pass too.

**Example.** `two_way_sort(list(range(16, 0, -1)), 2)` -> `(list(range(1, 17)), 4)`: 8 runs, then 4, 2, 1

**Big O.** 1 + ceil(log2(N / M)) passes over the data, each costing O(N / B) I/Os.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-2-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-2-2)

In [ ]:
# 8.2.2 Two-way external mergesort
def two_way_sort(data, M):
    raise NotImplementedError

In [ ]:
def test_ex_8_2_2():
    assert two_way_sort(list(range(16, 0, -1)), 2) == (list(range(1, 17)), 4)
    data = shuffled(10000, 1)
    srt, passes = two_way_sort(data, 100)
    assert srt == sorted(data) and passes == 8


test_ex_8_2_2()
print('8.2.2 passed')

### 8.2.3 · Merge K sorted arrays with a heap  `Core`

**Task.** Write `k_way_merge(lists)` with a heap of `(value, list_index, position)` entries, without `heapq.merge`.

**Example.** `k_way_merge([[1, 4, 9], [2, 3], [0, 10]])` -> `[0, 1, 2, 3, 4, 9, 10]`

**Big O.** O(N log K) for N total items in K lists.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-2-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-2-3)

In [ ]:
# 8.2.3 Merge K sorted arrays with a heap
def k_way_merge(lists):
    raise NotImplementedError

In [ ]:
def test_ex_8_2_3():
    assert k_way_merge([[1, 4, 9], [2, 3], [0, 10]]) == [0, 1, 2, 3, 4, 9, 10] and k_way_merge([]) == []


test_ex_8_2_3()
print('8.2.3 passed')

### 8.2.4 · M/B-way mergesort  `Core`

**Task.** Write `multiway_sort(data, M, B)` that makes runs of M items and then merges up to `M // B - 1` runs at a time (one input block per run plus one output block). Return `(sorted_list, passes)`.

**Example.** N = 10,000, M = 100, B = 10 -> 100 runs merged 9 at a time: passes `4` (versus 8 for two-way)

**Big O.** 1 + ceil(log_{M/B}(N / M)) passes. The wider the fan-in, the fewer passes over the disk.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-2-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-2-4)

In [ ]:
# 8.2.4 M/B-way mergesort
def multiway_sort(data, M, B):
    raise NotImplementedError

In [ ]:
def test_ex_8_2_4():
    data = shuffled(10000, 1)
    srt, passes = multiway_sort(data, 100, 10)
    assert srt == sorted(data) and passes == 4


test_ex_8_2_4()
print('8.2.4 passed')

### 8.2.5 · Predict the passes  `Core`

**Task.** Write `passes_two_way(N, M)` and `passes_multiway(N, M, B)` from the formulas, and check them against 8.2.2 and 8.2.4.

**Example.** `passes_two_way(10**4, 100)` -> `8`; `passes_multiway(10**4, 100, 10)` -> `4`

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-2-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-2-5)

In [ ]:
# 8.2.5 Predict the passes
def passes_two_way(N, M):
    raise NotImplementedError


def passes_multiway(N, M, B):
    raise NotImplementedError

In [ ]:
def test_ex_8_2_5():
    assert passes_multiway(10**4, 100, 10) == 4 and passes_two_way(10**4, 100) == 8


test_ex_8_2_5()
print('8.2.5 passed')

### 8.2.6 · Count the I/Os of an external sort  `Practice`

**Task.** Write `external_sort_ios(N, M, B)`: every pass reads and writes the whole data set, which is N / B blocks each way.

**Example.** `external_sort_ios(10**4, 100, 10)` -> `8000` (4 passes x 2 x 1000 blocks)

**Big O.** O((N / B) log_{M/B}(N / M)) I/Os, the known optimum for sorting in external memory.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-2-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-2-6)

In [ ]:
# 8.2.6 Count the I/Os of an external sort
def external_sort_ios(N, M, B):
    raise NotImplementedError

In [ ]:
def test_ex_8_2_6():
    assert external_sort_ios(10**4, 100, 10) == 8000


test_ex_8_2_6()
print('8.2.6 passed')

### 8.2.7 · Sort a real file that "does not fit"  `Practice`

**Task.** Write `sort_file(src, dst, max_items)` that sorts a file of one integer per line using at most `max_items` numbers in memory: write sorted runs to temporary files, then stream them through `heapq.merge` into dst.

**Example.** a file of 1,000 shuffled numbers with `max_items=100` -> dst holds them sorted, via 10 temporary run files

**Big O.** O(N log N) CPU, two passes over the disk. The same code sorts files far larger than RAM.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-2-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-2-7)

In [ ]:
# 8.2.7 Sort a real file that "does not fit"
def sort_file(src, dst, max_items):
    raise NotImplementedError

In [ ]:
def test_ex_8_2_7(tmp_path):
    src, dst = tmp_path / "in.txt", tmp_path / "out.txt"
    src.write_text("\n".join(map(str, shuffled(1000, 2))) + "\n")
    assert sort_file(src, dst, 100) == 10
    assert [int(x) for x in dst.read_text().split()] == list(range(1000))


test_ex_8_2_7(pathlib.Path(tempfile.mkdtemp()))
print('8.2.7 passed')